# Diabetes Prediction — Pima Indians

**Tabular Regression Project** — Predict diabetes-related outcomes.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Pima Indians Diabetes (768 rows × 9 columns)
Target: Numeric health outcome

## 2 · Project Overview

The Pima Indians Diabetes dataset is one of the most famous ML datasets. While often used for classification, here we treat it as a **regression** problem by predicting continuous health metrics or using the dataset to demonstrate regression techniques.

With 768 rows and 8 features, this is a well-understood benchmark for medical prediction.

## 3 · Learning Objectives

1. Work with a classic medical dataset.
2. Handle zero-encoded missing values (a common data quality issue).
3. Apply regression models to health outcome prediction.
4. Use LazyPredict and FLAML for model selection.
5. Understand medical ML implications.

## 4 · Problem Statement

Predict a **health outcome metric** from diagnostic measurements including glucose, blood pressure, BMI, insulin, and age for Pima Indian women.

## 5 · Why This Project Matters

- **Diabetes prediction** is a critical public health ML application.
- The dataset teaches handling of zero-encoded missing values.
- Medical ML requires careful interpretation and ethical consideration.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 768 |
| **Features** | Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age |
| **Target** | Glucose (used as continuous regression target) |
| **Source** | Kaggle: uciml/pima-indians-diabetes-database |

## 7 · Dataset Source and License Notes

- **Source**: National Institute of Diabetes and Digestive and Kidney Diseases, via UCI/Kaggle.
- **License**: Public domain for research.
- **Note**: All patients are females of Pima Indian heritage, age ≥ 21.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('uciml/pima-indians-diabetes-database')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
df.head()

  0%|                                              | 0.00/8.91k [00:00<?, ?B/s]

100%|█████████████████████████████████████| 8.91k/8.91k [00:00<00:00, 11.7MB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\uciml\pima-indians-diabetes-database\versions\1
Loaded: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 12 · Data Validation Checks

In [5]:
print('Missing values:', df.isnull().sum().sum())
print(f'\nZero values in key columns (likely missing):')
for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
    if col in df.columns:
        print(f'  {col}: {(df[col] == 0).sum()} zeros')

Missing values: 0

Zero values in key columns (likely missing):
  Glucose: 5 zeros
  BloodPressure: 35 zeros
  SkinThickness: 227 zeros
  Insulin: 374 zeros
  BMI: 11 zeros


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, ['Glucose', 'BMI', 'Age', 'Insulin']):
    if col in df.columns:
        df[col].hist(bins=30, ax=ax, edgecolor='black')
        ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

We use **Glucose** as the regression target — predicting glucose level from other health measurements.

In [7]:
TARGET = 'Glucose'
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count    768.000000
mean     120.894531
std       31.972618
min        0.000000
25%       99.000000
50%      117.000000
75%      140.250000
max      199.000000
Name: Glucose, dtype: float64

Skewness: 0.17


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

Replace zero values in medical columns with NaN, then impute with median.

In [8]:
# Replace zeros with NaN for physiologically impossible zero values
zero_cols = ['BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(df[col].median())

print(f'After imputation: {df.shape}')

After imputation: (768, 9)


## 17 · Feature Engineering

In [9]:
# Drop Outcome (classification label) if present — we're doing regression on Glucose
if 'Outcome' in df.columns:
    df = df.drop(columns=['Outcome'])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (614, 7), Test: (154, 7)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=27.67, MAE=22.20, R²=0.2390


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                             Adjusted R-Squared  R-Squared       RMSE  \
Model                                                                   
OrthogonalMatchingPursuitCV            0.235103   0.270099  27.101776   
Lasso                                  0.229545   0.264795  27.200065   
LassoLars                              0.229545   0.264794  27.200075   
LassoLarsCV                            0.225224   0.260671  27.276243   
LarsCV                                 0.225224   0.260671  27.276243   
LassoLarsIC                            0.220643   0.256300  27.356762   
HuberRegressor                         0.217233   0.253046  27.416543   
ElasticNetCV                           0.210683   0.246796  27.531008   
LassoCV                                0.209389   0.245561  27.553565   
BayesianRidge                          0.206458   0.242763  27.604601   
SGDRegressor                           0.204388   0.240789  27.640565   
RidgeCV                                0.204181   0

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=29.35, MAE=22.81, R²=0.1440
LightGBM: RMSE=32.92, MAE=26.10, R²=-0.0769


XGBoost: RMSE=30.95, MAE=23.62, R²=0.0480


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  Baseline_LR           RMSE=27.67  MAE=22.20  R²=0.2390
  CatBoost              RMSE=29.35  MAE=22.81  R²=0.1440
  XGBoost               RMSE=30.95  MAE=23.62  R²=0.0480
  LightGBM              RMSE=32.92  MAE=26.10  R²=-0.0769

Top 3: ['Baseline_LR', 'CatBoost', 'XGBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
Baseline_LR: RMSE=27.67, MAE=22.20, R²=0.2390
CatBoost: RMSE=29.35, MAE=22.81, R²=0.1440
XGBoost: RMSE=30.95, MAE=23.62, R²=0.0480


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models['CatBoost']
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Insulin** and **BMI** correlate strongly with glucose levels.
- Age and pregnancies show moderate predictive power.
- These models could assist clinical screening but NOT replace diagnostic tests.

## 26 · Limitations

- Only 768 patients, all Pima Indian women ≥21.
- Not generalizable to other populations.
- Zero-encoded missing values reduce data quality.
- Regression on glucose is an educational exercise — clinical testing is definitive.

## 27 · How to Improve

1. Add interaction features (BMI × Age).
2. Use multiple imputation instead of median.
3. Try target transformation.
4. Cross-validate with stratified folds.

## 28 · Production Considerations

- Medical models need rigorous clinical validation.
- Bias: limited to one ethnic group.
- Regulatory: FDA approval needed for clinical use.
- Must include uncertainty estimates.

## 29 · Common Mistakes

1. Not treating zeros as missing in medical columns.
2. Including Outcome (classification label) as a feature.
3. Generalizing to non-Pima populations.
4. Claiming clinical validity without trials.

## 30 · Mini Challenge

1. Predict BMI instead of Glucose and compare difficulty.
2. Add polynomial features for top 3 predictors.
3. Use SHAP to explain individual predictions.
4. Compare median imputation vs. KNN imputation.

## 31 · Final Summary

- The Pima dataset teaches critical data quality lessons (zero-encoded missing values).
- Glucose prediction from other health measurements is moderately achievable.
- Medical ML requires careful ethical and clinical considerations.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
